# 02 - Feature Engineering

This notebook demonstrates:
1. Computing technical indicators
2. Storing features in TimescaleDB
3. Building feature matrices for ML

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import matplotlib.pyplot as plt

from data_pipeline.loader import DataLoader
from features.engine import FeatureEngine
from features.technical import (
    compute_returns, compute_log_returns, compute_sma, compute_ema,
    compute_rsi, compute_macd, compute_bollinger_bands, compute_volatility
)
from features.registry import list_features

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load OHLCV Data

In [ ]:
loader = DataLoader()
df = loader.load_ohlcv('AAPL', start_date='2020-01-01')
print(f'Loaded {len(df)} rows')
df.head()

## 2. Compute Individual Indicators

In [ ]:
# Returns
df['returns'] = compute_returns(df)
df['log_returns'] = compute_log_returns(df)

# Moving averages
df['sma_20'] = compute_sma(df, window=20)
df['sma_50'] = compute_sma(df, window=50)
df['ema_12'] = compute_ema(df, span=12)
df['ema_26'] = compute_ema(df, span=26)

# RSI
df['rsi_14'] = compute_rsi(df, window=14)

# Volatility
df['volatility_20'] = compute_volatility(df, window=20)

# MACD
macd_df = compute_macd(df)
df['macd'] = macd_df['macd']
df['macd_signal'] = macd_df['signal']
df['macd_hist'] = macd_df['histogram']

# Bollinger Bands
bb_df = compute_bollinger_bands(df)
df['bb_upper'] = bb_df['upper']
df['bb_lower'] = bb_df['lower']

df.tail()

## 3. Visualize Indicators

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Price + MAs + Bollinger
recent = df.iloc[-200:]
axes[0].plot(recent.index, recent['close'], label='Close', linewidth=1)
axes[0].plot(recent.index, recent['sma_20'], label='SMA(20)', linewidth=0.8)
axes[0].plot(recent.index, recent['sma_50'], label='SMA(50)', linewidth=0.8)
axes[0].fill_between(recent.index, recent['bb_upper'], recent['bb_lower'], alpha=0.1)
axes[0].set_title('Price + Moving Averages + Bollinger Bands')
axes[0].legend()

# RSI
axes[1].plot(recent.index, recent['rsi_14'], linewidth=1)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('RSI(14)')
axes[1].set_ylim(0, 100)

# MACD
axes[2].plot(recent.index, recent['macd'], label='MACD', linewidth=1)
axes[2].plot(recent.index, recent['macd_signal'], label='Signal', linewidth=1)
axes[2].bar(recent.index, recent['macd_hist'], alpha=0.3, width=1)
axes[2].set_title('MACD')
axes[2].legend()

# Volatility
axes[3].plot(recent.index, recent['volatility_20'], linewidth=1)
axes[3].set_title('20-Day Rolling Volatility')

plt.tight_layout()
plt.show()

## 4. Use Feature Engine for Batch Processing

In [ ]:
# List available features
print('Available features:', list_features())

# Compute all features using the engine
engine = FeatureEngine()
feature_df = engine.compute_features(df)
print(f'\nFeature matrix shape: {feature_df.shape}')
print(f'Feature columns: {list(feature_df.columns)}')
feature_df.tail()

## 5. Store Features in Database

In [ ]:
# Compute and store features for AAPL
engine.compute_and_store('AAPL', start_date='2020-01-01')
print('Features stored successfully!')

# Verify by loading back
matrix = engine.get_feature_matrix('AAPL', ['returns', 'rsi_14', 'volatility_20'], '2024-01-01')
matrix.tail()